参考链接：  https://zhuanlan.zhihu.com/p/453787908
出于种种原因，我们有时候需要在函数外部得到函数内的局部变量。但是，由于Python中作用域的搜索顺序（"链式作用域"结构（chain scope）：子对象会一级一级地向上寻找所有父对象的变量），这一点通常是无法实现的。

In [1]:
def f1():
    n=999
    def f2():
        print(n)
    
    return f2

result = f1()
result()

999


在上面的代码中，函数f2就被包括在函数f1内部，这时f1内部的所有局部变量，对f2都是可见的。但是反过来就不行，f2内部的局部变量，对f1就是不可见的。这就是开头说到的，Python语言特有的作用域搜索顺序。所以，父对象的所有变量，对子对象都是可见的，反之则不成立。

既然f2可以读取f1中的局部变量，那么只要把f2作为返回值，我们不就可以在f1外部读取它的内部变量了吗?

# 闭包概念
上一部分代码中的f2函数，就是闭包。

在上面的实例中，有一个外层函数的局部变量 n，有一个内层函数 f2，f2 里面可以访问到 n 变量，那这f2就是一个闭包

闭包的一个定义和两个作用：

定义：闭包就是能够读取外部函数内的变量的函数。（前面已经讲解过）

作用1：闭包是将外层函数内的局部变量和外层函数的外部连接起来的一座桥梁。（下一部分讲解）

作用2：将外层函数的变量持久地保存在内存中。（下一部分讲解）

## （一）读取函数内部的变量

在这个例子里，我们想要一个给content加tag的功能，但是具体的tag_name是什么样子的要根据实际需求来定，对外部调用的接口已经确定，就是add_tag(content)。如果按照面向接口方式实现，我们会先把add_tag写成接口，指定其参数和返回类型，然后分别去实现a和b的add_tag。
但是在闭包的概念中，add_tag就是一个函数，它需要tag_name和content两个参数，只不过tag_name这个参数是打包带走的。所以一开始时就可以告诉我怎么打包，然后带走就行。

In [3]:
def tag(tag_name):
    def add_tag(content):
        return "<{0}>{1}</{0}>".format(tag_name, content)
    return add_tag
    
content = 'Hello'

add_tag = tag('a')
print(add_tag(content))
# <a>Hello</a>

add_tag = tag('b')
print(add_tag(content))
# <b>Hello</b>

<a>Hello</a>
<b>Hello</b>


## （二）让函数内部的局部变量始终保持在内存中

怎么来理解这句话呢？一般来说，函数内部的局部变量在这个函数运行完以后，就会被Python的垃圾回收机制从内存中清除掉。如果我们希望这个局部变量能够长久的保存在内存中，那么就可以用闭包来实现这个功能。

In [4]:
def create(pos=None):
    if pos is None:
        pos = [0,0]

    def go(direction, step):
        new_x = pos[0]+direction[0]*step
        new_y = pos[1]+direction[1]*step
        
        pos[0] = new_x
        pos[1] = new_y
        
        return pos
    
    
    return go

player = create()
print(player([1,0],10))
print(player([0,1],20))
print(player([-1,0],10))

[10, 0]
[10, 20]
[0, 20]


它一共运行了三次，第一次是沿X轴前进了10来到[10,0]，第二次是沿Y轴前进了20来到 [10, 20],，第三次是反方向沿X轴退了10来到[0, 20]。

这证明了，函数create中的局部变量pos一直保存在内存中，并没有在create调用后被自动清除。

为什么会这样呢？原因就在于create是go的父函数，而go被赋给了一个全局变量，这导致go始终在内存中，而go的存在依赖于create，因此create也始终在内存中，不会在调用结束后，被垃圾回收机制（garbage collection）回收。

这个时候，闭包使得函数的实例对象的内部变量，变得很像一个类的实例对象的属性，可以一直保存在内存中，并不断的对其进行运算。

## 判断一个函数是不是闭包

判断一个函数是不是闭包，可以查看它的closure属性。如果该函数是闭包，查看该属性将会返回一个cell对象组成的tuple。如果我们分别对每个cell对象查看其cell_contents属性，返回的内容就是闭包引用的自由变量的值。

In [7]:
def add(x,y):
    def f(z):
        return x+y+z
    return f

d = add(5,6)
print(d(9))
print(d(1))

d.__closure__

20
12


(<cell at 0x000001B6E6FEEEC0: int object at 0x00007FFC7F7C83A8>,
 <cell at 0x000001B6E6FEF460: int object at 0x00007FFC7F7C83C8>)

In [ ]:
# 闭包的__closure__方法，可以查看每个cell对象的内容。
for i in d.__closure__:
    print(i.cell_contents)

5
6


cell_contents解释了局部变量在脱离函数后仍然可以在函数之外被访问的原因，因为变量被存储在cell_contents中了。